# End-to-end workflow for building the training dataset

In [8]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import json
from pathlib import Path
from scipy.ndimage import zoom
from scipy.interpolate import RegularGridInterpolator
import torch

SCENE_PATH  = Path("../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC.nc")

# ── Band lists ────────────────────────────────────────────────────────────────
SAR_BANDS = [
    'nersc_sar_primary',    # HH
    'nersc_sar_secondary',  # HV
]

AMSR2_BANDS = [
    'btemp_6_9h',  'btemp_6_9v',
    'btemp_18_7h', 'btemp_18_7v',
    'btemp_36_5h', 'btemp_36_5v',
    'btemp_89_0h', 'btemp_89_0v',
]

ERA5_BANDS = [
    'u10m_rotated', 'v10m_rotated',
    't2m', 'skt', 'tcwv', 'tclw',
]

ANCILLARY_BANDS = ['sar_grid_incidenceangle', 'distance_map']

ALL_BANDS = SAR_BANDS + AMSR2_BANDS + ERA5_BANDS

## Create assembly harness and produce the two training GeoParquet tables
### Grab the list of training scenes from S3.
Iterate through each scene, and do the following:
- Create empty chip and patch tables for the file being processed.
- Call the data loader, which reads the file and iteratively returns chips. On each chip:
    - Generate embeddings
    - Compute patch-level ancillary features
    - Process labels
    - Assemble embeddings, labels, and ancillary features into table rows. Remember there are two tables, so that’s one row per chip, and 1024 per patch.
    - Append rows to the chip / patch tables
- Once all chips are processed, write the chip and patch tables to S3 as Geoparquet files. Name the tables with the Scene ID.
- Move on to the next file


In [5]:
import boto3
import json

BUCKET = "prescient-ice-data"
S3_PREFIX = "training_data/ai4arctic/raw_train/"
STATS_KEY  = "training_data/ai4arctic/statistics/dataset_stats.json"

session = boto3.Session(profile_name="spk_data")
s3 = session.client("s3")
response = s3.get_object(Bucket=BUCKET, Key=STATS_KEY)
stats    = json.loads(response['Body'].read().decode('utf-8'))

BAND_MEANS = {var: stats[var]['mean'] for var in ALL_BANDS if var in stats}

print(f"Loaded stats for {len(BAND_MEANS)} bands from S3")

Loaded stats for 16 bands from S3


### Open the NetCDF and read everything into memory

In [11]:
# ── Open scene once ───────────────────────────────────────────────────────────
ds = xr.open_dataset(SCENE_PATH, engine='netcdf4')

print("Scene dimensions:")
print(dict(ds.sizes))
print(f"\nSAR shape: {ds['nersc_sar_primary'].shape}")
print(f"AMSR2 shape: {ds['btemp_6_9h'].shape}")
print(f"ERA5 shape: {ds['u10m_rotated'].shape}")
print(f"GCP points: {ds.sizes.get('sar_grid_points', 'not found')}")

Scene dimensions:
{'sar_lines': 10006, 'sar_samples': 10458, 'sar_grid_points': 441, '2km_grid_lines': 200, '2km_grid_samples': 209, 'polygon_codes': 27}

SAR shape: (10006, 10458)
AMSR2 shape: (200, 209)
ERA5 shape: (200, 209)
GCP points: 441


In [12]:
# ── Read SAR into memory ──────────────────────────────────────────────────────
sar_h, sar_w = ds['nersc_sar_primary'].shape

sar = {
    var: ds[var].values.astype(np.float32)
    for var in SAR_BANDS
}

print(f"SAR loaded: {sar_h} x {sar_w} pixels")

# ── Valid mask: land = distance_map code 0, nodata = NaN in SAR ───────────────
# Compute BEFORE any substitution, as spec requires
distance_map = ds['distance_map'].values.astype(np.float32)
is_land      = (distance_map == 0)
is_nodata    = np.isnan(sar['nersc_sar_primary'])
valid_mask   = ~is_land & ~is_nodata   # True = valid pixel

print(f"Valid pixels: {valid_mask.sum():,} / {valid_mask.size:,} "
      f"({100*valid_mask.mean():.1f}%)")
print(f"Land pixels:   {is_land.sum():,}")
print(f"Nodata pixels: {is_nodata.sum():,}")

SAR loaded: 10006 x 10458 pixels
Valid pixels: 93,892,767 / 104,642,748 (89.7%)
Land pixels:   10,005,348
Nodata pixels: 879,874


In [13]:
# ── Substitute land/nodata pixels with band mean ──────────────────────────────
# After substitution, these pixels are exactly zero in Clay's normalised space
# (because mean - mean = 0 after z-score normalisation)
for var in SAR_BANDS:
    fill = BAND_MEANS.get(var, 0.0)
    sar[var][~valid_mask] = fill

print("SAR substitution done")

SAR substitution done


### Resample ancillaries to SAR resolution

In [14]:
def resample_to_sar(arr, target_h, target_w, order=1):
    """
    Bilinear resample arr to (target_h, target_w).
    order=1 is bilinear, order=0 is nearest-neighbour.
    """
    zoom_h = target_h / arr.shape[0]
    zoom_w = target_w / arr.shape[1]
    return zoom(arr.astype(np.float32), (zoom_h, zoom_w), order=order)


# ── Resample AMSR2 ────────────────────────────────────────────────────────────
amsr2 = {}
for var in AMSR2_BANDS:
    raw        = ds[var].values.astype(np.float32)
    resampled  = resample_to_sar(raw, sar_h, sar_w)
    # Substitute invalid pixels
    fill = BAND_MEANS.get(var, 0.0)
    resampled[~valid_mask] = fill
    amsr2[var] = resampled

print(f"AMSR2 resampled: {list(amsr2.values())[0].shape}")

# ── Resample ERA5 ─────────────────────────────────────────────────────────────
era5 = {}
for var in ERA5_BANDS:
    raw       = ds[var].values.astype(np.float32)
    resampled = resample_to_sar(raw, sar_h, sar_w)
    fill = BAND_MEANS.get(var, 0.0)
    resampled[~valid_mask] = fill
    era5[var] = resampled

print(f"ERA5 resampled:  {list(era5.values())[0].shape}")

# ── Resample distance_map and incidence angle ─────────────────────────────────
# distance_map is already at SAR resolution — no resample needed
# incidence angle is on the GCP sparse grid — handled in Section 4 via interpolation

print("Ancillary resampling complete")

AMSR2 resampled: (10006, 10458)
ERA5 resampled:  (10006, 10458)
Ancillary resampling complete


### GCP interpolation for lat/lon and incidence angle

In [15]:
# ── Read GCPs ─────────────────────────────────────────────────────────────────
n_gcps = ds.dims['sar_grid_points']   # dynamic, typically 441 (21x21)
gcp_side = int(np.sqrt(n_gcps))       # 21

gcp_lines  = ds['sar_grid_line'].values        # pixel row indices
gcp_samps  = ds['sar_grid_sample'].values      # pixel col indices
gcp_lats   = ds['sar_grid_latitude'].values
gcp_lons   = ds['sar_grid_longitude'].values
gcp_angles = ds['sar_grid_incidenceangle'].values

print(f"GCPs: {n_gcps} ({gcp_side}x{gcp_side} grid)")
print(f"Line range: {gcp_lines.min():.0f} → {gcp_lines.max():.0f}")
print(f"Sample range: {gcp_samps.min():.0f} → {gcp_samps.max():.0f}")

GCPs: 441 (21x21 grid)
Line range: 0 → 10005
Sample range: 0 → 10457


/var/folders/w3/lvdn2w6d72ngld7k70v66yg40000gn/T/ipykernel_11282/1134226930.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_gcps = ds.dims['sar_grid_points']   # dynamic, typically 441 (21x21)


In [16]:
# ── Build 2D interpolators ────────────────────────────────────────────────────
# GCPs are on a regular grid in line/sample space — reshape to 2D first
lines_2d  = gcp_lines.reshape(gcp_side, gcp_side)
samps_2d  = gcp_samps.reshape(gcp_side, gcp_side)
lats_2d   = gcp_lats.reshape(gcp_side, gcp_side)
lons_2d   = gcp_lons.reshape(gcp_side, gcp_side)
angles_2d = gcp_angles.reshape(gcp_side, gcp_side)

# Row/col axes of the interpolator — use the unique line/sample values
row_axis = lines_2d[:, 0]    # one value per row
col_axis = samps_2d[0, :]    # one value per col

interp_lat   = RegularGridInterpolator((row_axis, col_axis), lats_2d,   method='linear')
interp_lon   = RegularGridInterpolator((row_axis, col_axis), lons_2d,   method='linear')
interp_angle = RegularGridInterpolator((row_axis, col_axis), angles_2d, method='linear')

def get_chip_geo(row_center, col_center):
    """Return (lat, lon, incidence_angle) for a pixel coordinate."""
    pt  = np.array([[row_center, col_center]])
    lat = float(interp_lat(pt))
    lon = float(interp_lon(pt))
    ang = float(interp_angle(pt))
    return lat, lon, ang

# Quick sanity check
lat0, lon0, ang0 = get_chip_geo(sar_h // 2, sar_w // 2)
print(f"Scene centre: lat={lat0:.3f}, lon={lon0:.3f}, incidence={ang0:.2f}°")

TypeError: only 0-dimensional arrays can be converted to Python scalars